In [1]:
!pip install --upgrade gdown
import gdown
folder_url = 'https://drive.google.com/drive/folders/1s5W0Nfz9NEN8gP14tge8GuouUDXs2Ssq'
gdown.download_folder(folder_url, quiet=False, use_cookies=False)

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


Retrieving folder contents


Processing file 1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF full-shape.zip
Processing file 1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p partial.zip
Processing file 1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM rotate.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF
From (redirected): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF&confirm=t&uuid=1a726523-3f62-40bf-892a-f707b49b2fce
To: /content/AffordanceNet/full-shape.zip
100%|██████████| 558M/558M [00:11<00:00, 50.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p
From (redirected): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p&confirm=t&uuid=9dbb6215-67c5-4f01-9ad3-cb02c4275f9c
To: /content/AffordanceNet/partial.zip
100%|██████████| 2.19G/2.19G [00:24<00:00, 88.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM
To: /content/AffordanceNet/rotate.zip
100%|██████████| 1.15M/1.15M [00:00<00:00, 10.9MB/s]
Download completed


['/content/AffordanceNet/full-shape.zip',
 '/content/AffordanceNet/partial.zip',
 '/content/AffordanceNet/rotate.zip']

In [2]:
!unzip -o /content/AffordanceNet/full-shape.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/partial.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/rotate.zip -d /content/AffordanceNet/
!rm /content/AffordanceNet/*.zip

Archive:  /content/AffordanceNet/full-shape.zip
  inflating: /content/AffordanceNet/full_shape_train_data.pkl  
  inflating: /content/AffordanceNet/full_shape_val_data.pkl  
Archive:  /content/AffordanceNet/partial.zip
  inflating: /content/AffordanceNet/partial_train_data.pkl  
  inflating: /content/AffordanceNet/partial_val_data.pkl  
Archive:  /content/AffordanceNet/rotate.zip
  inflating: /content/AffordanceNet/rotate_val_data.pkl  


In [3]:
!pip install torch torchvision tqdm numpy scikit-learn

In [4]:
# === CELL 4: CONFIGURATION — CHANGE THIS BETWEEN RUNS ===

# ┌─────────────────────────────────────────────────────────┐
# │  SET THIS FLAG:                                         │
# │    False = XYZ-only baseline (INPUT_DIM=3)              │
# │    True  = Full model with normals+curvature (DIM=7)    │
# └─────────────────────────────────────────────────────────┘
USE_GEOMETRIC_FEATURES = True     # ← CHANGE THIS BETWEEN RUNS

NUM_EPOCHS = 50                    # 50 epochs for both experiments
BATCH_SIZE = 8
LEARNING_RATE = 5e-4
K_NEIGHBORS = 20

INPUT_DIM = 7 if USE_GEOMETRIC_FEATURES else 3
EXPERIMENT_NAME = "geo7d" if USE_GEOMETRIC_FEATURES else "xyz3d"
SAVE_PATH = f"/content/output/best_{EXPERIMENT_NAME}.pth"

print(f"{'='*60}")
print(f"  EXPERIMENT: {EXPERIMENT_NAME.upper()}")
print(f"  Input dim: {INPUT_DIM}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Geometric features: {USE_GEOMETRIC_FEATURES}")
print(f"  Save path: {SAVE_PATH}")
print(f"{'='*60}")

  EXPERIMENT: GEO7D
  Input dim: 7
  Epochs: 50
  Geometric features: True
  Save path: /content/output/best_geo7d.pth


In [5]:
# === CELL 5: Load Data and Configure Split ===

import pickle
import numpy as np
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm import tqdm
import os

with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
with open("/content/AffordanceNet/full_shape_val_data.pkl", "rb") as f:
    val_data = pickle.load(f)

AFFORDANCE_NAMES = train_data[0]['affordance']

# Category → affordances mapping
cat_affs = defaultdict(set)
for s in train_data:
    cat = s['semantic class']
    for aff in AFFORDANCE_NAMES:
        if (s['full_shape']['label'][aff].flatten() > 0.5).any():
            cat_affs[cat].add(aff)

# Affordance selection (same as experiment 10)
SELECTED_CLASSES = ['grasp', 'contain', 'wrap_grasp', 'pourable']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_CLASSES = len(SELECTED_CLASSES)

# Zero-shot split (same as experiment 10)
TEST_CATEGORIES = ['Mug', 'Vase', 'Scissors']
all_cats = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_cats]

def has_positive(sample):
    for idx in SELECTED_INDICES:
        aff = AFFORDANCE_NAMES[idx]
        if (sample['full_shape']['label'][aff].flatten() > 0.5).any():
            return True
    return False

train_split = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES and has_positive(s)]
val_split   = [s for s in val_data   if s['semantic class'] not in TEST_CATEGORIES and has_positive(s)]
test_split  = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES and has_positive(s)]

# Class weights
total_pos = np.zeros(NUM_CLASSES)
total_all = np.zeros(NUM_CLASSES)
for s in train_split:
    label_dict = s['full_shape']['label']
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        mask = label_dict[aff].flatten()
        total_pos[i] += (mask > 0.5).sum()
        total_all[i] += len(mask)
total_neg = total_all - total_pos
pos_weights = np.clip(total_neg / (total_pos + 1), 1.0, 50.0)

print(f"Train: {len(train_split)}, Val: {len(val_split)}, Test: {len(test_split)}")
print(f"pos_weights: {[f'{w:.1f}' for w in pos_weights]}")

Train: 3550, Val: 498, Test: 917
pos_weights: ['38.3', '16.3', '50.0', '50.0']


In [6]:
# === CELL 6: Precompute Features ===

def compute_geometric_features(points):
    """Returns (N, 7): [x,y,z, nx,ny,nz, curvature]"""
    N = points.shape[0]
    if N < 10:
        return np.hstack([points, np.zeros((N, 3)), np.zeros((N, 1))]).astype(np.float32)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pts = torch.from_numpy(points).float().to(device)
    k = min(30, N - 1)
    dists = torch.cdist(pts, pts)
    _, idx = dists.topk(k=k + 1, dim=-1, largest=False)
    idx = idx[:, 1:]
    neighbors = pts[idx]
    centered = neighbors - neighbors.mean(dim=1, keepdim=True)
    cov = torch.bmm(centered.transpose(1, 2), centered) / k
    eigvals, eigvecs = torch.linalg.eigh(cov)
    curv = eigvals[:, 0] / (eigvals.sum(dim=1) + 1e-10)
    normals = eigvecs[:, :, 0]
    # Orient outward
    centroid = pts.mean(dim=0)
    outward = pts - centroid
    flip = (normals * outward).sum(dim=1) < 0
    normals[flip] = -normals[flip]
    return np.hstack([points, normals.cpu().numpy(), curv.cpu().numpy()[:, None]]).astype(np.float32)


def precompute(data_list, desc=""):
    augmented = []
    for s in tqdm(data_list, desc=desc):
        pts = s['full_shape']['coordinate'].astype(np.float32)
        if USE_GEOMETRIC_FEATURES:
            feats = compute_geometric_features(pts)  # (N, 7)
        else:
            feats = pts.copy()                        # (N, 3) — XYZ only
        augmented.append({
            'features': feats,
            'labels': s['full_shape']['label'],
            'category': s['semantic class']
        })
    return augmented


train_aug = precompute(train_split, "Train features")
val_aug   = precompute(val_split,   "Val features")
test_aug  = precompute(test_split,  "Test features")
print(f"Feature dim: {train_aug[0]['features'].shape[1]}")

Test features: 100%|██████████| 917/917 [00:02<00:00, 426.13it/s]

Feature dim: 7


In [7]:
# === CELL 7: Oversample ===

class_sample_counts = np.zeros(NUM_CLASSES)
for s in train_aug:
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        if (s['labels'][aff].flatten() > 0.5).any():
            class_sample_counts[i] += 1

max_count = class_sample_counts.max()
oversample_factors = {}
for i in range(NUM_CLASSES):
    factor = int(np.ceil(max_count / (class_sample_counts[i] + 1)))
    oversample_factors[i] = max(1, min(factor, 8))

oversampled_train = []
for s in train_aug:
    oversampled_train.append(s)
    best_factor = 1
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        if (s['labels'][aff].flatten() > 0.5).any():
            best_factor = max(best_factor, oversample_factors[i])
    for _ in range(best_factor - 1):
        oversampled_train.append(s)

print(f"Oversampled: {len(train_aug)} → {len(oversampled_train)}")

Oversampled: 3550 → 9002


In [8]:
# === CELL 8: Dataset ===

class GeoAffordanceDataset(Dataset):
    def __init__(self, data, selected_indices, affordance_names,
                 num_points=2048, occlusion_prob=0.0, crop_angle=120):
        self.data = data
        self.selected_indices = selected_indices
        self.affordance_names = affordance_names
        self.num_points = num_points
        self.occlusion_prob = occlusion_prob
        self.crop_angle = crop_angle

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        pts = sample['features'].copy()
        label_dict = sample['labels']
        dim = pts.shape[1]  # 3 or 7

        masks = []
        for aff_idx in self.selected_indices:
            aff = self.affordance_names[aff_idx]
            masks.append((label_dict[aff].flatten() > 0.5).astype(np.float32))
        labels = np.stack(masks, axis=1)

        # Occlusion
        if np.random.random() < self.occlusion_prob:
            coords = pts[:, :3]
            centroid = coords.mean(axis=0)
            centered = coords - centroid
            norms = np.linalg.norm(centered, axis=1)
            norms[norms < 1e-8] = 1e-8
            dirs = centered / norms[:, None]
            theta = np.random.uniform(0, 2 * np.pi)
            phi = np.random.uniform(0, np.pi)
            view = np.array([np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)])
            visible = (dirs @ view) > np.cos(np.radians(self.crop_angle / 2))
            if visible.sum() >= 64:
                pts = pts[visible]
                labels = labels[visible]

        # Normalize XYZ
        coords = pts[:, :3].copy()
        centroid = coords.mean(axis=0)
        coords -= centroid
        md = np.max(np.linalg.norm(coords, axis=1))
        if md > 1e-8: coords /= md
        pts[:, :3] = coords

        # Subsample (preserve positives)
        N = len(pts)
        if N >= self.num_points:
            pos_mask = labels.sum(axis=1) > 0.5
            pos_idx = np.where(pos_mask)[0]
            neg_idx = np.where(~pos_mask)[0]
            if len(pos_idx) <= self.num_points:
                n_neg = self.num_points - len(pos_idx)
                neg_chosen = np.random.choice(neg_idx, min(n_neg, len(neg_idx)), replace=False) if len(neg_idx) > 0 else np.array([], dtype=int)
                chosen = np.concatenate([pos_idx, neg_chosen])
            else:
                chosen = np.random.choice(pos_idx, self.num_points, replace=False)
            if len(chosen) < self.num_points:
                extra = np.random.choice(N, self.num_points - len(chosen), replace=True)
                chosen = np.concatenate([chosen, extra])
        else:
            chosen = np.random.choice(N, self.num_points, replace=True)

        return (torch.tensor(pts[chosen], dtype=torch.float32),
                torch.tensor(labels[chosen], dtype=torch.float32))

In [9]:
# === CELL 9: Model ===

def knn(x, k=20):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pw = -xx - inner - xx.transpose(2, 1)
    return pw.topk(k=k, dim=-1)[1]

def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None: idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx_flat = (idx + idx_base).view(-1)
    x_t = x.transpose(2, 1).contiguous().view(B * N, -1)
    feat = x_t[idx_flat].view(B, N, k, C)
    x_rep = x.transpose(2, 1).view(B, N, 1, C).expand(-1, -1, k, -1)
    return torch.cat([feat - x_rep, x_rep], dim=3).permute(0, 3, 1, 2).contiguous()

class DGCNNAblation(nn.Module):
    """DGCNN that works with both 3D and 7D input."""
    def __init__(self, num_classes, input_dim=3, k=20):
        super().__init__()
        self.k = k
        self.input_dim = input_dim
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_dim * 2, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.head = nn.Sequential(
            nn.Conv1d(512, 256, 1, bias=False),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Conv1d(128, num_classes, 1))

    def forward(self, x):
        B, C, N = x.shape
        # First layer: KNN always on XYZ (first 3 dims)
        xyz = x[:, :3, :]
        idx1 = knn(xyz, k=self.k)
        edge = get_graph_feature(x, k=self.k, idx=idx1)
        x1 = self.conv1(edge).max(dim=-1)[0]
        edge = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge).max(dim=-1)[0]
        edge = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge).max(dim=-1)[0]
        x_local = torch.cat([x1, x2, x3], dim=1)             # (B, 256, N)
        x_global = x_local.max(dim=-1, keepdim=True)[0].expand(-1, -1, N)
        out = self.head(torch.cat([x_local, x_global], dim=1))
        return out.permute(0, 2, 1)

In [10]:
# === CELL 10: Loss + Sampler ===

class FocalBCEDiceLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0, smooth=1.0):
        super().__init__()
        self.register_buffer('pos_weight', pos_weight)
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, pred, target):
        bce_raw = nn.functional.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pw = self.pos_weight.unsqueeze(0).unsqueeze(0)
        weight = torch.where(target > 0.5, pw, torch.ones_like(pw))
        pt = torch.sigmoid(pred)
        pt = torch.where(target > 0.5, pt, 1 - pt)
        focal = (1 - pt) ** self.gamma
        bce = (focal * weight * bce_raw).mean()

        pred_sig = torch.sigmoid(pred)
        dice = 0.0
        for c in range(pred.size(2)):
            inter = (pred_sig[..., c] * target[..., c]).sum()
            union = pred_sig[..., c].sum() + target[..., c].sum()
            dice += 1 - (2. * inter + self.smooth) / (union + self.smooth)
        return bce + dice / pred.size(2)


class BalancedSampler(Sampler):
    def __init__(self, dataset, batch_size, num_classes, affordance_names, selected_indices):
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.class_to_idx = defaultdict(list)
        for i, s in enumerate(dataset.data):
            for c_idx, aff_idx in enumerate(selected_indices):
                aff = affordance_names[aff_idx]
                if (s['labels'][aff].flatten() > 0.5).any():
                    self.class_to_idx[c_idx].append(i)
        self.num_samples = len(dataset)
        self.num_batches = self.num_samples // batch_size

    def __iter__(self):
        pools = {c: np.random.permutation(v).tolist() for c, v in self.class_to_idx.items()}
        batches = []
        for _ in range(self.num_batches):
            batch, used = [], set()
            for c in range(self.num_classes):
                if not pools[c]:
                    pools[c] = np.random.permutation(self.class_to_idx[c]).tolist()
                idx = pools[c].pop()
                if idx not in used:
                    batch.append(idx)
                    used.add(idx)
            filler = np.random.permutation(self.num_samples)
            for idx in filler:
                if len(batch) >= self.batch_size: break
                if int(idx) not in used:
                    batch.append(int(idx))
                    used.add(int(idx))
            while len(batch) < self.batch_size:
                batch.append(np.random.randint(self.num_samples))
            batches.append(batch)
        np.random.shuffle(batches)
        for b in batches: yield b

    def __len__(self): return self.num_batches

In [11]:
# === CELL 11: Create Loaders ===

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = GeoAffordanceDataset(oversampled_train, SELECTED_INDICES, AFFORDANCE_NAMES,
                                 num_points=2048, occlusion_prob=0.5, crop_angle=120)
val_ds   = GeoAffordanceDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                 num_points=2048, occlusion_prob=0.0)
test_clean_ds = GeoAffordanceDataset(test_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                      num_points=2048, occlusion_prob=0.0)
test_occ_ds   = GeoAffordanceDataset(test_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                      num_points=2048, occlusion_prob=1.0, crop_angle=120)

sampler = BalancedSampler(train_ds, BATCH_SIZE, NUM_CLASSES, AFFORDANCE_NAMES, SELECTED_INDICES)
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_clean_loader = DataLoader(test_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_occ_loader   = DataLoader(test_occ_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Sanity check
pts_check, lab_check = next(iter(train_loader))
print(f"Input shape: {pts_check.shape} → permuted: (B, {INPUT_DIM}, 2048)")
for c in range(NUM_CLASSES):
    n = (lab_check[:,:,c] > 0.5).sum().item()
    print(f"  {SELECTED_CLASSES[c]}: {n} positive points in batch")

Input shape: torch.Size([8, 2048, 7]) → permuted: (B, 7, 2048)
  grasp: 765 positive points in batch
  contain: 865 positive points in batch
  wrap_grasp: 268 positive points in batch
  pourable: 444 positive points in batch


In [12]:
# === CELL 12: Train ===

model = DGCNNAblation(num_classes=NUM_CLASSES, input_dim=INPUT_DIM, k=K_NEIGHBORS).to(device)
with torch.no_grad():
    model.head[-1].bias.data.fill_(1.0)

pw_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
criterion = FocalBCEDiceLoss(pw_tensor, gamma=2.0).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

os.makedirs("/content/output", exist_ok=True)
best_val = 0.0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for pts, labels in tqdm(train_loader, desc=f"[{EXPERIMENT_NAME}] Epoch {epoch+1}/{NUM_EPOCHS}"):
        pts = pts.permute(0, 2, 1).to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(pts), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # Validate
    model.eval()
    iou_c = torch.zeros(NUM_CLASSES, device=device)
    cnt_c = torch.zeros(NUM_CLASSES, device=device)
    with torch.no_grad():
        for pts, labels in val_loader:
            pts = pts.permute(0,2,1).to(device)
            labels = labels.to(device)
            pred = (torch.sigmoid(model(pts)) > 0.5).float()
            for b in range(pts.size(0)):
                for c in range(NUM_CLASSES):
                    gt = labels[b,:,c]
                    if gt.sum() > 0:
                        inter = (pred[b,:,c] * gt).sum()
                        union = (pred[b,:,c] + gt).clamp(0,1).sum()
                        iou_c[c] += inter / (union + 1e-8)
                        cnt_c[c] += 1

    avg_iou = iou_c / (cnt_c + 1e-8)
    miou = avg_iou.mean().item()
    history.append({'epoch': epoch+1, 'loss': avg_loss, 'miou': miou,
                    'per_class': avg_iou.cpu().tolist()})

    cls = ', '.join(f"{SELECTED_CLASSES[i]}={avg_iou[i].item():.3f}" for i in range(NUM_CLASSES))
    print(f"  Loss {avg_loss:.4f} | Val mIoU {miou:.4f} | {cls}")

    scheduler.step()
    if miou > best_val:
        best_val = miou
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✅ Saved (best={best_val:.4f})")

print(f"\n🏆 [{EXPERIMENT_NAME}] Best Val mIoU: {best_val:.4f}")

[geo7d] Epoch 1/50: 100%|██████████| 1125/1125 [02:34<00:00,  7.27it/s]


  Loss 1.2389 | Val mIoU 0.1124 | grasp=0.081, contain=0.165, wrap_grasp=0.089, pourable=0.114
  ✅ Saved (best=0.1124)


[geo7d] Epoch 2/50: 100%|██████████| 1125/1125 [02:41<00:00,  6.99it/s]


  Loss 1.1917 | Val mIoU 0.1288 | grasp=0.095, contain=0.144, wrap_grasp=0.084, pourable=0.192
  ✅ Saved (best=0.1288)


[geo7d] Epoch 3/50: 100%|██████████| 1125/1125 [02:42<00:00,  6.93it/s]


...

  Loss 1.0985 | Val mIoU 0.2116 | grasp=0.142, contain=0.301, wrap_grasp=0.107, pourable=0.296


[geo7d] Epoch 49/50: 100%|██████████| 1125/1125 [02:42<00:00,  6.93it/s]


  Loss 1.0931 | Val mIoU 0.2092 | grasp=0.138, contain=0.298, wrap_grasp=0.111, pourable=0.289


[geo7d] Epoch 50/50: 100%|██████████| 1125/1125 [02:42<00:00,  6.94it/s]


  Loss 1.0905 | Val mIoU 0.2129 | grasp=0.147, contain=0.299, wrap_grasp=0.109, pourable=0.296

🏆 [geo7d] Best Val mIoU: 0.2178


In [13]:
# === CELL 13: Four-Condition Evaluation ===

model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

def evaluate(loader, desc):
    iou_c = np.zeros(NUM_CLASSES)
    cnt_c = np.zeros(NUM_CLASSES)
    with torch.no_grad():
        for pts, labels in tqdm(loader, desc=desc):
            pts = pts.permute(0,2,1).to(device)
            pred = (torch.sigmoid(model(pts)) > 0.5).cpu().numpy()
            labels = labels.numpy()
            for b in range(pred.shape[0]):
                for c in range(NUM_CLASSES):
                    gt = labels[b,:,c]
                    if gt.sum() > 0:
                        inter = (pred[b,:,c] * gt).sum()
                        union = np.clip(pred[b,:,c] + gt, 0, 1).sum()
                        iou_c[c] += inter / (union + 1e-8)
                        cnt_c[c] += 1
    avg = iou_c / (cnt_c + 1e-8)
    m = avg.mean()
    print(f"\n{desc} — mIoU: {m:.4f}")
    for i, n in enumerate(SELECTED_CLASSES):
        print(f"  {n}: {avg[i]:.4f} (n={int(cnt_c[i])})")
    return m, avg

seen_clean_ds = GeoAffordanceDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                      num_points=2048, occlusion_prob=0.0)
seen_occ_ds   = GeoAffordanceDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                      num_points=2048, occlusion_prob=1.0, crop_angle=120)
seen_clean_ldr = DataLoader(seen_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
seen_occ_ldr   = DataLoader(seen_occ_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\n{'='*60}")
print(f"  RESULTS: {EXPERIMENT_NAME.upper()} ({INPUT_DIM}D input, {NUM_EPOCHS} epochs)")
print(f"{'='*60}")

r1, _ = evaluate(seen_clean_ldr,    "Seen / Clean")
r2, _ = evaluate(seen_occ_ldr,      "Seen / Occluded")
r3, _ = evaluate(test_clean_loader, "Unseen / Clean")
r4, _ = evaluate(test_occ_loader,   "Unseen / Occluded")

print(f"\n{'='*50}")
print(f"  SUMMARY — {EXPERIMENT_NAME.upper()}")
print(f"{'='*50}")
print(f"{'':20s} | {'Clean':>8s} | {'Occluded':>8s}")
print(f"{'-'*20}-+-{'-'*8}-+-{'-'*8}")
print(f"{'Seen':20s} | {r1:8.4f} | {r2:8.4f}")
print(f"{'Unseen (zero-shot)':20s} | {r3:8.4f} | {r4:8.4f}")
print(f"{'='*50}")


  RESULTS: GEO7D (7D input, 50 epochs)


Seen / Clean: 100%|██████████| 63/63 [00:05<00:00, 10.53it/s]



Seen / Clean — mIoU: 0.2178
  grasp: 0.1496 (n=177)
  contain: 0.3362 (n=361)
  wrap_grasp: 0.1050 (n=57)
  pourable: 0.2806 (n=82)


Seen / Occluded: 100%|██████████| 63/63 [00:05<00:00, 10.68it/s]



Seen / Occluded — mIoU: 0.1883
  grasp: 0.1359 (n=117)
  contain: 0.3015 (n=249)
  wrap_grasp: 0.1105 (n=43)
  pourable: 0.2054 (n=46)


Unseen / Clean: 100%|██████████| 115/115 [00:10<00:00, 10.70it/s]



Unseen / Clean — mIoU: 0.1039
  grasp: 0.0168 (n=176)
  contain: 0.1873 (n=817)
  wrap_grasp: 0.0394 (n=861)
  pourable: 0.1720 (n=811)


Unseen / Occluded: 100%|██████████| 115/115 [00:10<00:00, 10.80it/s]


Unseen / Occluded — mIoU: 0.1038
  grasp: 0.0559 (n=99)
  contain: 0.1490 (n=488)
  wrap_grasp: 0.0648 (n=623)
  pourable: 0.1455 (n=459)

  SUMMARY — GEO7D
                     |    Clean | Occluded
---------------------+----------+---------
Seen                 |   0.2178 |   0.1883
Unseen (zero-shot)   |   0.1039 |   0.1038


In [14]:
# === CELL 14: Save Training History (for plotting later) ===

import json
hist_path = f"/content/output/history_{EXPERIMENT_NAME}.json"
with open(hist_path, 'w') as f:
    json.dump(history, f)
print(f"Training history saved to {hist_path}")

# Print best results per epoch
print(f"\n--- Top 5 epochs by Val mIoU ({EXPERIMENT_NAME}) ---")
sorted_hist = sorted(history, key=lambda x: x['miou'], reverse=True)[:5]
for h in sorted_hist:
    print(f"  Epoch {h['epoch']}: mIoU={h['miou']:.4f}, Loss={h['loss']:.4f}")


Training history saved to /content/output/history_geo7d.json

--- Top 5 epochs by Val mIoU (geo7d) ---
  Epoch 29: mIoU=0.2178, Loss=1.1163
  Epoch 47: mIoU=0.2157, Loss=1.0850
  Epoch 26: mIoU=0.2149, Loss=1.1095
  Epoch 45: mIoU=0.2148, Loss=1.0984
  Epoch 32: mIoU=0.2148, Loss=1.1031
